In [ ]:
import azure.cognitiveservices.speech as speechsdk
from tkinter import Tk
from tkinter.filedialog import askopenfilename
import threading

# Azure Speech Service Subscription Info
azure_speech_key = "d38a1f7591004c4595b50845b5e2198f"  # Replace with your Azure Speech key
azure_service_region = "eastus"  # Replace with your Azure service region

# Create a threading event to manage session completion
recognition_done = threading.Event()

def recognize_speech_from_audio_file(audio_filename):
    """Recognize speech from an audio file using Azure and save the transcription to a text file."""
    
    # Create a speech configuration with subscription key and region
    speech_config = speechsdk.SpeechConfig(subscription=azure_speech_key, region=azure_service_region)

    # Set recognition language to Korean (change to your preferred language if needed)
    speech_config.speech_recognition_language = "ko-KR"

    # Set up audio input from a file (MP3, WAV, etc.)
    audio_config = speechsdk.AudioConfig(filename=audio_filename)

    # Create a speech recognizer for the audio file input
    speech_recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_config)

    # This will store the full transcription
    transcription = []

    # Define a callback to handle recognized speech and append it to the transcription list
    def handle_recognized(evt):
        if evt.result.text:
            text = evt.result.text.strip()
            if not text.endswith(('.', '!', '?')):  # Ensure proper sentence termination
                text += '.'
            print(f"Recognized: {text}")
            transcription.append(text)

    # Define a callback to handle session stopped and save the transcription to a file
    def stop_recognition(evt):
        print("Recognition stopped.")
        # Save the transcription to a text file
        with open("azure_transcription.txt", "w", encoding="utf-8") as file:
            file.write("\n".join(transcription))  # Each phrase on a new line
        print("Transcription saved to 'azure_transcription.txt'.")
        # Signal that recognition is done
        recognition_done.set()

    # Connect the event handlers
    speech_recognizer.recognized.connect(handle_recognized)
    speech_recognizer.session_stopped.connect(stop_recognition)
    speech_recognizer.canceled.connect(stop_recognition)

    # Start file-based recognition
    print(f"Transcribing audio file: {audio_filename}")
    speech_recognizer.start_continuous_recognition()

    # Wait for recognition to complete
    recognition_done.wait()

if __name__ == "__main__":
    # Use tkinter file dialog to select the audio file
    Tk().withdraw()  # Prevents a full GUI window from popping up
    audio_file = askopenfilename(title="Select Audio File", filetypes=[("Audio Files", "*.wav *.mp3")])
    
    if audio_file:
        recognize_speech_from_audio_file(audio_file)
    else:
        print("No file selected.")